In [ ]:
import pickle
import gzip

scenario_name = "Reference"

with gzip.open(f"results/{scenario_name}/model.pkl", "rb") as f:
    model = pickle.load(f)

Steps:
1) deep_merge_yaml.py to create mapping yaml file based on defaults, project, etc.
2) cell below to export results using cims_export_template.csv

In [ ]:
### Export results ###

%reload_ext autoreload
%autoreload 2

import sys
sys.path.insert(1, "sources/mapping")
import yaml_partial_match
from itertools import product
from datetime import datetime
import pandas as pd

def get_param_func(param, node, year, context=None, sub_context=None, tech=None, energy=None, service=None, ghg_gas=None, ghg_type=None):
    value_temp = 0

    try:
        if param == "quantity_provided":
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_provided_by_total()
        
        elif param == "quantity_requested" and energy == "Total":
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_requested_by_total()
        elif param == "quantity_requested" and energy != "Total" and service:
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_requested_by_energy_service()[energy][service]
        elif param == "quantity_requested" and energy != "Total":
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_requested_by_energy()[energy]
        
        elif param == "quantity_distributed" and energy == "Total":
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_distributed_by_total()
        elif param == "quantity_distributed" and energy != "Total" and service:
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_distributed_by_energy_service()[energy][service]
        elif param == "quantity_distributed" and energy != "Total":
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_distributed_by_energy()[energy]
                
        elif "emissions_" in param and service is not None:
            value_temp = CIMS.emissions.sum_emissions_by_service(
                    model, 
                    param=param,
                    node=node,
                    year=year,
                    child=service).sum_emissions_by_ghg_type()[ghg_gas][ghg_type]
        elif "emissions_" in param and energy is not None:
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_emissions_by_energy_ghg_type()[energy][ghg_gas][ghg_type]
        elif "emissions_" in param:
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,).sum_emissions_by_ghg_type()[ghg_gas][ghg_type]
                
        else:
            value_temp = model.get_param(
                param=param,
                node=node,
                year=year,
                context=context,
                sub_context=sub_context,
                tech=tech,)
    
    except KeyError:
        pass
        
    return value_temp


def safe_float(value, default=0.0):
    """Safely convert any value to float."""
    try:
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            return float(value.strip().replace(',', ''))
        return default
    except (ValueError, TypeError):
        return default


def load_map(path):
    """Load the variable mapping YAML."""
    with open(path) as f:
        return yaml.safe_load(f)
    

def load_template(path="test_data.csv"):
    """Load the output template CSV defining which results to export."""
    df = pd.read_csv(path)
    # Normalize column names
    df.columns = [c.strip().lower() for c in df.columns]
    return df


def extract_model_results(model, variable_map, template):
    """
    Loop through output template rows and extract results
    using the variable_map definitions and get_param_func().
    """
    results = []

    for _, row in template.iterrows():
        row_vars = row.to_dict()

        try:
            var_info = variable_map[row_vars["category"]][row_vars["param"]]
        except KeyError:
            print(f"Warning: Category '{row_vars['category']}' or param '{row_vars['param']}' not found in variable_map")
            continue
        
        region_list = []
        for r in var_info["region"].find_from_key(row_vars["region"]):
            if r["include"] == "CAN":
                region_list.append("CIMS." + r["include"])
            else:
                region_list.append("CIMS.CAN." + r["include"])

        sector_list = []
        for s in var_info["sector"].find_from_key(row_vars["sector"]):
            includes = s.get("include")
            excludes = s.get("service_exclude")
            # Get multiplier from unit_multiplier (set by find_from_key)
            multiplier = float(s.get("multiplier", 1.0))

            # Normalize to lists
            includes = includes if isinstance(includes, list) else [includes]
            excludes = excludes if isinstance(excludes, list) else [excludes] if excludes else [None]

            # Make one tuple per combination
            for incl in includes:
                for excl in excludes:
                    sector_list.append((incl, excl, multiplier))

        energy_list = []
        for e in var_info["energy"].find_from_key(row_vars["energy"]):
            energy_list.append((e.get("include"), e.get("sector_exclude")))

        tech_list = []
        for t in var_info["tech"].find_from_key(row_vars["tech"]):
            tech_list.append(t["include"])

        ghg_gas_list = []
        for g in var_info["ghg_gas"].find_from_key(row_vars["context"]):
            ghg_gas_list.append(g["include"])

        ghg_type_list = []
        for y in var_info["ghg_type"].find_from_key(row_vars["sub_context"]):
            ghg_type_list.append(y["include"])

        value_total = 0

        lists = [
            region_list or [None],
            sector_list or [(None, None, None)],
            energy_list or [(None, None)],
            tech_list or [None],
            ghg_gas_list or [None],
            ghg_type_list or [None],
        ]

        for r, (n_incl, service_excl, mult), (e_incl, sector_excl), t, g, y in product(*lists):
            # print(row_vars["category"], row_vars["param"], r, n_incl, e_incl, t, g, y)
            value_incl = 0
            value_excl = 0

            ghg_gas = None
            ghg_type = None
            context = None
            sub_context = None

            sector_excl_list = sector_excl if isinstance(sector_excl, list) else [sector_excl]

            if sector_excl and any(ext == n_incl for ext in sector_excl_list):
                n_incl = None
            elif n_incl and "{energy}" in n_incl:
                n_incl = n_incl.format(energy=e_incl.replace("CIMS.Generic Fuels.", ""))
            elif n_incl and "{region}" in n_incl:
                n_incl = n_incl.format(region=r)

            if e_incl:
                e_incl = e_incl.format(region=r)

            if service_excl:
                service_excl = service_excl.format(region=r)
            
            if any(ext in row_vars["param"] for ext in ["emissions_"]):
                ghg_gas = g
                ghg_type = y
                context = None
                sub_context = None
            elif any(ext in row_vars["param"] for ext in ["tax"]):
                ghg_gas = None
                ghg_type = None
                context = g
                sub_context = y
            elif any(ext in row_vars["param"] for ext in ["multiplier_price"]):
                ghg_gas = None
                ghg_type = None
                context = e_incl
                sub_context = None
            
            value_incl = get_param_func(
                param=row_vars["param"], 
                node=n_incl, 
                year=str(row_vars["time"]),
                context=context,
                sub_context=sub_context,
                tech=t,
                energy=e_incl,
                ghg_gas=ghg_gas,
                ghg_type=ghg_type,
            )
            
            if service_excl:
                value_excl = get_param_func(
                    param=row_vars["param"], 
                    node=n_incl, 
                    year=str(row_vars["time"]),
                    context=context,
                    sub_context=sub_context,
                    tech=t,
                    energy=e_incl,
                    service=service_excl,
                    ghg_gas=ghg_gas,
                    ghg_type=ghg_type,
                )
            
            # Safely convert to float and apply multiplier
            value_incl_num = safe_float(value_incl)
            value_excl_num = safe_float(value_excl)
            
            value_total += value_incl_num * mult
            value_total -= value_excl_num * mult
            # print(f"{row_vars['param']}: {n_incl}: {value_incl_num:,.1f}")
        # print(f"{row_vars['region']}: {row_vars['time']}: {row_vars['category']}: {row_vars['param']}: {row_vars['sector']}: {row_vars['energy']}: {row_vars['tech']}: {row_vars['context']}: {row_vars['sub_context']}: ({value_total:,.1f})")
        
        # Add to results
        results.append({
            **row_vars,
            'value': value_total
        })
    
    return results

variable_map = yaml_partial_match.load_yaml("sources/mapping/cur_variable.yaml")
template = load_template("sources/mapping/cims_export_template.csv")

results = extract_model_results(model, variable_map, template)

df = pd.DataFrame(results)
df.to_csv(f"results/{scenario_name}/results_{scenario_name}_{datetime.now().strftime("%y%m%d")}.csv", index=False)